<a href="https://colab.research.google.com/github/Prince-tech-debug/LLM-Exploration/blob/main/Parallel_LLM_comparision_using_OPENROUTER_FREE_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install litellm streamlit pandas matplotlib plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 113.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 106.2 MB/s eta 0:00:00


In [3]:
import gradio as gr
import litellm
import time
import os
from google.colab import userdata

# Set your API Key
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

def benchmark_models(prompt, model_a, model_b):
    results = []

    for model_name in [model_a, model_b]:
        start_time = time.time()
        try:
            # Step 1: Attempt the primary model
            response = litellm.completion(
                model=f"openrouter/{model_name}",
                messages=[{"role": "user", "content": prompt}],
                timeout=20
            )

            latency = round(time.time() - start_time, 2)
            content = response['choices'][0]['message']['content']
            stats = f"✅ Success | ⏱️ {latency}s | 🎫 {response['usage']['total_tokens']} tokens"
            results.append((content, stats))

        except Exception as e:
            # Step 2: Auto-Fallback to any working free model if primary fails
            try:
                fallback_response = litellm.completion(
                    model="openrouter/openrouter/free",
                    messages=[{"role": "user", "content": prompt}]
                )
                content = fallback_response['choices'][0]['message']['content']
                results.append((content, "🔄 Fallback Active (Original Model Offline)"))
            except:
                results.append((f"❌ Critical Error: {str(e)}", "N/A"))

    return results[0][0], results[0][1], results[1][0], results[1][1]

# --- Build the Gradio UI ---
with gr.Blocks(theme=gr.themes.Monochrome()) as app:
    gr.Markdown("# 🏆 LLM Benchmarking Suite")

    with gr.Row():
        # Updated with the most stable 2026 IDs
        model_input_a = gr.Dropdown(
            choices=["google/gemma-3-27b-it:free", "meta-llama/llama-3.3-70b-instruct:free", "openrouter/free"],
            label="Model A (Stable)", value="google/gemma-3-27b-it:free"
        )
        model_input_b = gr.Dropdown(
            choices=["mistralai/mistral-small-3.1-24b-instruct:free", "nvidia/nemotron-3-nano-30b-a3b:free", "openrouter/free"],
            label="Model B (Fast)", value="openrouter/free"
        )

    user_prompt = gr.Textbox(label="Prompt", value="Compare the time complexity of QuickSort vs MergeSort.")
    btn = gr.Button("🚀 Run Comparison", variant="primary")

    with gr.Row():
        with gr.Column():
            output_a = gr.Markdown()
            stats_a = gr.Label(label="Performance A")
        with gr.Column():
            output_b = gr.Markdown()
            stats_b = gr.Label(label="Performance B")

    btn.click(fn=benchmark_models, inputs=[user_prompt, model_input_a, model_input_b], outputs=[output_a, stats_a, output_b, stats_b])

if __name__ == "__main__":
    app.launch()

/tmp/ipykernel_522/2201717207.py:43: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Monochrome()) as app:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://aa217bccc82ad37d07.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
